# Hunyuan3D-2 + texture
Cài đặt: Settings -> Accelerator = GPU T4, Internet = On. Add Data -> dataset chứa ảnh Pixar (.jpg/.png).
Chạy hết các ô. Kết quả: /kaggle/working/h3d_tex_out/<tên ảnh>.glb (khoảng 45 giây mỗi ảnh).

In [ ]:
open('/kaggle/working/run.sh','w').write('set -x\ncd /kaggle/working\nnvidia-smi --query-gpu=name,memory.total --format=csv\nnvcc --version | tail -2\npython --version; python -c "import torch;print(torch.__version__, torch.version.cuda)"\ngit clone -q https://github.com/Tencent-Hunyuan/Hunyuan3D-2 H3D\npip -q install diffusers transformers accelerate omegaconf einops trimesh rembg onnxruntime pyyaml opencv-python-headless safetensors xatlas pybind11 fast_simplification pytorch-lightning realesrgan basicsr 2>&1 | tail -3\ncd H3D/hy3dgen/texgen/custom_rasterizer && python setup.py install 2>&1 | tail -15\ncd ../differentiable_renderer && python setup.py install 2>&1 | tail -15\ncd /kaggle/working\npython run_tex.py\n')
open('/kaggle/working/run_tex.py','w',encoding='utf-8').write('import sys, os, glob, time, types, gc, torch\ntry:\n    import pymeshlab\nexcept Exception:\n    m = types.ModuleType("pymeshlab"); m.MeshSet = object; sys.modules["pymeshlab"] = m\nsys.path.insert(0, "/kaggle/working/H3D")\nfrom PIL import Image\nfrom rembg import remove\nfrom hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline\n# nhận mọi ảnh .jpg/.jpeg/.png trong dataset đã gắn vào notebook (tên file = tên kết quả)\nfiles = sorted(f for f in glob.glob("/kaggle/input/**/*", recursive=True) if f.lower().endswith((".jpg", ".jpeg", ".png")))\nassert files, "không thấy ảnh nào trong /kaggle/input: hãy Add Data -> dataset chứa ảnh Pixar"\nnames = {os.path.splitext(os.path.basename(f))[0]: f for f in files}; print("ảnh:", list(names), flush=True)\nOUT = "/kaggle/working/h3d_tex_out"; os.makedirs(OUT, exist_ok=True)\npipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained("tencent/Hunyuan3D-2mini", subfolder="hunyuan3d-dit-v2-mini", variant="fp16")\nshapes = {}\nfor n in names:\n    im = remove(Image.open(names[n]).convert("RGB"))\n    mesh = pipe(image=im, num_inference_steps=30, octree_resolution=256, num_chunks=20000, generator=torch.manual_seed(2025), output_type="trimesh")[0]\n    mesh = mesh.simplify_quadric_decimation(face_count=40000)\n    shapes[n] = (im, mesh); print(n, "shape", len(mesh.faces), flush=True)\ndel pipe; gc.collect(); torch.cuda.empty_cache()\nfrom hy3dgen.texgen import Hunyuan3DPaintPipeline\npaint = Hunyuan3DPaintPipeline.from_pretrained("tencent/Hunyuan3D-2", subfolder="hunyuan3d-paint-v2-0-turbo")\ntry: paint.enable_model_cpu_offload()\nexcept Exception as e: print("no offload", e)\nfor n, (im, mesh) in shapes.items():\n    t = time.time()\n    try:\n        out = paint(mesh, image=im)\n        out.export(f"{OUT}/{n}.glb"); print(n, "textured", round(time.time() - t, 1), "s", flush=True)\n    except Exception as e:\n        import traceback; traceback.print_exc(); print(n, "FAIL", e, flush=True)\n    torch.cuda.empty_cache()\n')

In [ ]:
!bash /kaggle/working/run.sh 2>&1 | tail -40